# Importação e limpeza dos dados

Este notebook substitui a etapa de importação do script `import_csv.py`. Os CSVs são carregados e tratados como `pandas.DataFrame`, prontos para análise.

In [1]:
from pathlib import Path
import re
import unicodedata

import pandas as pd

PASTA_DADOS = Path("../data")
DATAS_SENTINELA = {pd.Timestamp("1900-01-01"), pd.Timestamp("2099-12-31")}

COLUNAS = {
    "clientes": [
        "id_cliente", "data_cadastro", "data_ultima_atualizacao_cadastro",
        "uf", "cidade", "faixa_etaria", "canal_aquisicao",
        "tier_clube", "optin_email", "optout",
    ],
    "interacoes": [
        "id_cliente", "ano_mes", "campanhas_enviadas",
        "campanhas_reativacao_recebidas", "campanhas_abertas", "cliques",
        "sessoes_site_app", "carrinhos_abandonados", "tickets_sac",
        "motivo_sac_principal", "ticket_resolvido",
    ],
    "pedidos": [
        "id_pedido", "id_cliente", "data_pedido", "canal", "categoria",
        "sku", "quantidade", "valor_unitario", "valor_desconto",
        "cupom_utilizado", "prazo_entrega_dias", "atraso_entrega_dias",
        "margem_bruta_pct",
    ],
}


In [2]:
ESTADOS_BRASIL = {
    "Acre": "AC", "Alagoas": "AL", "Amapá": "AP", "Amazonas": "AM",
    "Bahia": "BA", "Ceará": "CE", "Distrito Federal": "DF",
    "Espírito Santo": "ES", "Goiás": "GO", "Maranhão": "MA",
    "Mato Grosso": "MT", "Mato Grosso do Sul": "MS", "Minas Gerais": "MG",
    "Pará": "PA", "Paraíba": "PB", "Paraná": "PR", "Pernambuco": "PE",
    "Piauí": "PI", "Rio de Janeiro": "RJ", "Rio Grande do Norte": "RN",
    "Rio Grande do Sul": "RS", "Rondônia": "RO", "Roraima": "RR",
    "Santa Catarina": "SC", "São Paulo": "SP", "Sergipe": "SE", "Tocantins": "TO",
}

def sem_acentos(valor):
    valor = unicodedata.normalize("NFD", str(valor).casefold().strip())
    return re.sub(r"[\u0300-\u036f]", "", valor)

UF_PARA_SIGLA = {
    re.sub(r"[\s._-]+", "", sem_acentos(nome)): sigla
    for nome, sigla in ESTADOS_BRASIL.items()
} | {sigla.casefold(): sigla for sigla in ESTADOS_BRASIL.values()} | {"marahao": "MA"}

def normalizar_texto(serie):
    return serie.astype("string").str.normalize("NFD").str.replace(r"[\u0300-\u036f]", "", regex=True).str.upper()

def normalizar_uf(valor):
    if pd.isna(valor):
        return pd.NA
    chave = re.sub(r"[\s._-]+", "", sem_acentos(valor))
    return UF_PARA_SIGLA.get(chave, valor)

def ler_csv(nome):
    caminho = PASTA_DADOS / f"{nome}.csv"
    df = pd.read_csv(caminho, dtype="string", encoding="utf-8-sig")
    df.columns = df.columns.str.strip()
    if list(df.columns) != COLUNAS[nome]:
        raise ValueError(f"{caminho.name}: cabeçalho incompatível. Esperado: {COLUNAS[nome]}")
    return df.replace(r"^\s*$", pd.NA, regex=True)


In [3]:
# Carrega os arquivos brutos
clientes = ler_csv("clientes")
interacoes = ler_csv("interacoes")
pedidos_brutos = ler_csv("pedidos")

# Limpeza de clientes
clientes["uf"] = clientes["uf"].map(normalizar_uf).astype("string")
for coluna in ["cidade", "canal_aquisicao", "tier_clube"]:
    clientes[coluna] = normalizar_texto(clientes[coluna])
for coluna in ["data_cadastro", "data_ultima_atualizacao_cadastro"]:
    clientes[coluna] = pd.to_datetime(clientes[coluna], errors="raise")
for coluna in ["optin_email", "optout"]:
    clientes[coluna] = clientes[coluna].str.casefold().map({"true": True, "t": True, "sim": True, "s": True, "1": True, "false": False, "f": False, "nao": False, "não": False, "n": False, "0": False}).astype("boolean")

# Limpeza de interações
interacoes["motivo_sac_principal"] = normalizar_texto(interacoes["motivo_sac_principal"])
interacoes["ano_mes"] = pd.to_datetime(interacoes["ano_mes"], errors="raise")
for coluna in ["campanhas_enviadas", "campanhas_reativacao_recebidas", "campanhas_abertas", "cliques", "sessoes_site_app", "carrinhos_abandonados", "tickets_sac"]:
    interacoes[coluna] = pd.to_numeric(interacoes[coluna], errors="raise").astype("Int64")
interacoes["ticket_resolvido"] = interacoes["ticket_resolvido"].str.casefold().map({"true": True, "t": True, "sim": True, "s": True, "1": True, "false": False, "f": False, "nao": False, "não": False, "n": False, "0": False}).astype("boolean")


In [4]:
# Pedidos: separa o pedido (um por id_pedido) de seus itens
pedidos_brutos["canal"] = normalizar_texto(pedidos_brutos["canal"])
pedidos_brutos["categoria"] = normalizar_texto(pedidos_brutos["categoria"])
pedidos_brutos["data_pedido"] = pd.to_datetime(pedidos_brutos["data_pedido"], errors="raise")
for coluna in ["quantidade", "prazo_entrega_dias", "atraso_entrega_dias"]:
    pedidos_brutos[coluna] = pd.to_numeric(pedidos_brutos[coluna], errors="raise").astype("Int64")
for coluna in ["valor_unitario", "valor_desconto", "margem_bruta_pct"]:
    pedidos_brutos[coluna] = pd.to_numeric(pedidos_brutos[coluna].str.replace(",", ".", regex=False), errors="raise")
pedidos_brutos["cupom_utilizado"] = pedidos_brutos["cupom_utilizado"].str.casefold().map({"true": True, "t": True, "sim": True, "s": True, "1": True, "false": False, "f": False, "nao": False, "não": False, "n": False, "0": False}).astype("boolean")
pedidos_brutos["atraso_entrega_dias"] = pedidos_brutos["atraso_entrega_dias"].fillna(0)

# Datas sentinela são substituídas pela primeira data válida do mesmo pedido.
datas_validas = pedidos_brutos["data_pedido"].where(~pedidos_brutos["data_pedido"].isin(DATAS_SENTINELA))
pedidos_brutos["data_pedido"] = pedidos_brutos["data_pedido"].where(datas_validas.notna(), datas_validas.groupby(pedidos_brutos["id_pedido"]).transform("first"))

# Mantém somente pedidos e interações vinculados a clientes existentes.
ids_clientes_validos = set(clientes["id_cliente"])
interacoes = interacoes[interacoes["id_cliente"].isin(ids_clientes_validos)].copy()
pedidos_brutos = pedidos_brutos[pedidos_brutos["id_cliente"].isin(ids_clientes_validos)].copy()
pedidos_brutos = pedidos_brutos.dropna(subset=["data_pedido"])

pedidos = pedidos_brutos.drop_duplicates("id_pedido")[[
    "id_pedido", "id_cliente", "data_pedido", "canal", "cupom_utilizado",
    "prazo_entrega_dias", "atraso_entrega_dias",
]].copy()
pedido_itens = pedidos_brutos[[
    "id_pedido", "categoria", "sku", "quantidade", "valor_unitario",
    "valor_desconto", "margem_bruta_pct",
]].copy()


In [5]:
resumo = pd.DataFrame({
    "dataframe": ["clientes", "interacoes", "pedidos", "pedido_itens"],
    "linhas": [len(clientes), len(interacoes), len(pedidos), len(pedido_itens)],
})
resumo


,dataframe,linhas
0,clientes,12120
1,interacoes,322173
2,pedidos,78794
3,pedido_itens,212909


In [6]:
clientes.head()

,id_cliente,data_cadastro,data_ultima_atualizacao_cadastro,uf,cidade,faixa_etaria,canal_aquisicao,tier_clube,optin_email,optout
0,CLI100000,2025-01-11,2026-06-28,CE,SOBRAL,18-24,E-COMMERCE,BRONZE,True,False
1,CLI100001,2023-04-15,2023-09-08,PA,SANTAREM,25-34,E-COMMERCE,BRONZE,True,False
2,CLI100002,2025-07-10,2025-09-21,PA,SANTAREM,<NA>,LOJA FRANQUEADA,BRONZE,True,False
3,CLI100003,2019-06-22,2026-04-10,AM,MANAUS,45-54,LOJA FRANQUEADA,PRATA,True,False
4,CLI100004,2024-09-25,2026-05-30,MA,TIMON,25-34,APP,PRATA,True,False


In [7]:
interacoes.head()

,id_cliente,ano_mes,campanhas_enviadas,campanhas_reativacao_recebidas,campanhas_abertas,cliques,sessoes_site_app,carrinhos_abandonados,tickets_sac,motivo_sac_principal,ticket_resolvido
0,CLI100000,2025-04-01,4,0,1,0,9,2,0,<NA>,<NA>
1,CLI100000,2025-05-01,1,0,0,0,3,0,0,<NA>,<NA>
2,CLI100000,2025-06-01,2,0,0,0,5,1,0,<NA>,<NA>
3,CLI100000,2025-07-01,3,0,1,0,3,1,0,<NA>,<NA>
4,CLI100000,2025-08-01,7,0,2,0,4,0,0,<NA>,<NA>


In [8]:
pedidos.head()

,id_pedido,id_cliente,data_pedido,canal,cupom_utilizado,prazo_entrega_dias,atraso_entrega_dias
0,PED500001,CLI100000,2025-04-08,E-COMMERCE,True,2,0
3,PED500002,CLI100000,2025-09-12,E-COMMERCE,True,5,0
8,PED500003,CLI100000,2025-10-18,APP,False,5,0
12,PED500004,CLI100000,2026-03-21,APP,False,4,11
15,PED500005,CLI100000,2026-05-03,E-COMMERCE,False,4,0


In [9]:
pedido_itens.head()

,id_pedido,categoria,sku,quantidade,valor_unitario,valor_desconto,margem_bruta_pct
0,PED500001,BANHO E CORPO,SKU1809,1,42.66,13.16,0.6
1,PED500001,CABELOS,SKU1048,1,69.46,18.8,0.55
2,PED500001,CUIDADOS COM A PELE,SKU1251,1,198.23,37.58,0.62
3,PED500002,MAQUIAGEM,SKU1384,1,240.01,34.99,0.58
4,PED500002,BANHO E CORPO,SKU1955,2,130.63,89.5,0.6


In [ ]:
# Base analítica: uma linha por cliente, pronta para a etapa de clusterização.
DATA_REFERENCIA = pd.Timestamp("2026-06-30")

# Valor dos itens e valor final de cada pedido.
itens_valor = pedido_itens.assign(
    valor_bruto=lambda df: df["quantidade"] * df["valor_unitario"],
    valor_liquido=lambda df: df["quantidade"] * df["valor_unitario"] - df["valor_desconto"],
)
valor_por_pedido = itens_valor.groupby("id_pedido", as_index=False).agg(
    valor_bruto=("valor_bruto", "sum"),
    valor_desconto=("valor_desconto", "sum"),
    valor_liquido=("valor_liquido", "sum"),
)
pedidos_cluster = pedidos.merge(valor_por_pedido, on="id_pedido", how="left")

# Métricas de compra, preço, cupom e entrega.
pedidos_cluster["janela_12m"] = pedidos_cluster["data_pedido"] >= DATA_REFERENCIA - pd.DateOffset(months=12)
pedidos_cluster["janela_90d"] = pedidos_cluster["data_pedido"] > DATA_REFERENCIA - pd.Timedelta(days=90)
pedidos_cluster["janela_90d_anterior"] = (pedidos_cluster["data_pedido"] > DATA_REFERENCIA - pd.Timedelta(days=180)) & ~pedidos_cluster["janela_90d"]
pedidos_cluster["pct_desconto_pedido"] = pedidos_cluster["valor_desconto"] / pedidos_cluster["valor_bruto"]

compras_cliente = pedidos_cluster.groupby("id_cliente").agg(
    data_ultimo_pedido=("data_pedido", "max"),
    n_pedidos_12m=("janela_12m", "sum"),
    ticket_medio=("valor_liquido", "mean"),
    valor_total_liquido_12m=("valor_liquido", lambda s: s[pedidos_cluster.loc[s.index, "janela_12m"]].sum()),
    proporcao_pedidos_cupom=("cupom_utilizado", "mean"),
    desconto_medio_pct=("pct_desconto_pedido", "mean"),
    pct_pedidos_atrasados=("atraso_entrega_dias", lambda s: (s > 0).mean()),
    atraso_medio_dias=("atraso_entrega_dias", "mean"),
    pedidos_ultimos_90d=("janela_90d", "sum"),
    pedidos_90d_anteriores=("janela_90d_anterior", "sum"),
).reset_index()

intervalos = (pedidos_cluster[["id_cliente", "data_pedido"]].drop_duplicates().sort_values(["id_cliente", "data_pedido"])
    .groupby("id_cliente")["data_pedido"].apply(lambda s: s.diff().dt.days.mean())
    .rename("intervalo_medio_pedidos").reset_index())
compras_cliente = compras_cliente.merge(intervalos, on="id_cliente", how="left")
compras_cliente["recencia_dias"] = (DATA_REFERENCIA - compras_cliente.pop("data_ultimo_pedido")).dt.days
pedidos_ultimos_90d = compras_cliente.pop("pedidos_ultimos_90d")
pedidos_90d_anteriores = compras_cliente.pop("pedidos_90d_anteriores")
compras_cliente["variacao_frequencia_90d"] = (pedidos_ultimos_90d - pedidos_90d_anteriores) / pedidos_90d_anteriores.clip(lower=1)

# Preferência de categoria.
categorias_cliente = pedido_itens.assign(eh_presente=lambda df: df["categoria"].eq("PRESENTES E KITS")).groupby("id_pedido").agg(eh_presente=("eh_presente", "max")).reset_index()
categorias_cliente = pedidos[["id_pedido", "id_cliente"]].merge(categorias_cliente, on="id_pedido", how="left").groupby("id_cliente").agg(share_presentes_kits=("eh_presente", "mean")).reset_index()
n_categorias = pedidos[["id_pedido", "id_cliente"]].merge(pedido_itens[["id_pedido", "categoria"]], on="id_pedido", how="left").groupby("id_cliente")["categoria"].nunique().rename("n_categorias_compradas").reset_index()

# Métricas mensais de relacionamento e SAC.
interacoes_aux = interacoes.assign(
    tickets_nao_resolvidos=lambda df: df["tickets_sac"] * df["ticket_resolvido"].eq(False).fillna(False).astype(int)
)
engajamento_cliente = interacoes_aux.groupby("id_cliente").agg(
    campanhas_enviadas=("campanhas_enviadas", "sum"),
    campanhas_abertas=("campanhas_abertas", "sum"),
    cliques=("cliques", "sum"),
    sessoes_site_app=("sessoes_site_app", "mean"),
    carrinhos_abandonados=("carrinhos_abandonados", "mean"),
    tickets_sac=("tickets_sac", "sum"),
    tickets_nao_resolvidos=("tickets_nao_resolvidos", "sum"),
    meses_observados=("ano_mes", "nunique"),
).reset_index()
engajamento_cliente["taxa_abertura_email"] = engajamento_cliente.pop("campanhas_abertas") / engajamento_cliente["campanhas_enviadas"].replace(0, pd.NA)
engajamento_cliente["taxa_cliques"] = engajamento_cliente.pop("cliques") / engajamento_cliente["campanhas_enviadas"].replace(0, pd.NA)
engajamento_cliente["taxa_tickets_sac"] = engajamento_cliente.pop("tickets_sac") / engajamento_cliente["meses_observados"]
engajamento_cliente["taxa_tickets_nao_resolvidos"] = engajamento_cliente.pop("tickets_nao_resolvidos") / engajamento_cliente["meses_observados"]
engajamento_cliente = engajamento_cliente.drop(columns=["campanhas_enviadas", "meses_observados"])

# Junta todas as métricas e preserva todos os clientes cadastrados.
clientes_cluster = (clientes[["id_cliente", "data_cadastro"]]
    .merge(compras_cliente, on="id_cliente", how="left")
    .merge(categorias_cliente, on="id_cliente", how="left")
    .merge(n_categorias, on="id_cliente", how="left")
    .merge(engajamento_cliente, on="id_cliente", how="left"))
clientes_cluster["recencia_dias"] = clientes_cluster["recencia_dias"].fillna((DATA_REFERENCIA - clientes_cluster["data_cadastro"]).dt.days)
clientes_cluster = clientes_cluster.drop(columns="data_cadastro")

COLUNAS_CLUSTER = [
    "id_cliente", "recencia_dias", "n_pedidos_12m", "ticket_medio",
    "valor_total_liquido_12m", "intervalo_medio_pedidos",
    "variacao_frequencia_90d", "proporcao_pedidos_cupom",
    "desconto_medio_pct", "pct_pedidos_atrasados", "atraso_medio_dias",
    "taxa_abertura_email", "taxa_cliques", "sessoes_site_app",
    "carrinhos_abandonados", "taxa_tickets_sac",
    "taxa_tickets_nao_resolvidos", "share_presentes_kits",
    "n_categorias_compradas",
]
clientes_cluster = clientes_cluster[COLUNAS_CLUSTER]
clientes_cluster.to_csv(PASTA_DADOS / "clientes_cluster.csv", index=False)
clientes_cluster.head()